# Superstore Merge
Scott Person

## Setup the data folder location

In [0]:
data_folder='./data/'

## Python Imports

## Install a required component
This gets rid of an issue that Databricks was having importing the Excel file. Not sure if it's required by other environments.

In [0]:
!pip install openpyxl

In [0]:
import pandas as pd

## Import the Data

In [0]:
df = pd.read_excel(data_folder + 'Global_Superstore_Orders_2012_2015-1.xlsx')

## Get Statistics and Descriptions about the Data

In [0]:
df.head()

In [0]:
df.tail()

In [0]:
df.sample(10) # --> NOTE: This is an additional often useful method. Try it out. Then look up "pandas .sample() method" to learn what it does.

In [0]:
df.shape

In [0]:
df.info()

In [0]:
df.describe()

### Get Description of Object Columns

In [0]:
df.describe(include='object')

## File Merge

In [0]:
df_sheet2 = pd.read_excel(data_folder + 'Global_Superstore_Orders_2012_2015-1.xlsx', sheet_name=1)
df_sheet2.head()
df_sheet2.describe(include='object')

In [0]:
# Based on this, probably need to merge on Region. Would be nice if there was some indicator of what type of Person. Regional Manager?
df_sheet2.head()

In [0]:
df = df.merge(df_sheet2, how='left', on='Region')

In [0]:
df.head()

In [0]:
df.tail()

In [0]:
df.sample(10)

In [0]:
df.shape

In [0]:
df.info()

In [0]:
df.describe()

In [0]:
# Look for columns where data is missing
df.isna().sum()

In [0]:
# Return records with missing values
df[df.isna().any(axis=1)]

In [0]:
df[df['Person'].isna()]

# Cleaning Steps

## Setting Data Types

In [0]:
df.info()

### Postal Code
We need to change postal code to a string as it's not really a number that we perform operations on. Also unlike a number a postal code can have leading zeroes. We're going to pad that out while we're doing the conversion.

In [0]:

df["Postal Code"] = (
    df["Postal Code"]
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
    .astype("object")
)
df.dtypes

In [0]:
df.sample(10)

### Category Columns
Note that there is some opportunity to change a few columns to category type but I don't believe that we've covered that in this class so I'm going to defer that.

## Look for Missing Data

In [0]:
# Look for columns where data is missing
df.isna().sum()

In [0]:
df[(df['Postal Code'].isna()) & (df['Country'] == 'United States')]

### Missing Data Discussion

#### Postal Codes
In my experience postal codes are not standard across international schemes. Note in the cell immediately above I looked for missing postal codes with Country = United States. Since this returned no records, for the sake of this dataset we can safely assume that the blank postal codes are a reasonable value for other countries.

#### Person
Person requires more analysis. I'll investigate below. Because that dataset joins on region I'll check out what regions have empty Person columns.

In [0]:
df[df['Person'].isna()].groupby('Region').size().reset_index(name='Count').sort_values('Count', ascending=False)

All Canada so let's go back to the original data.

In [0]:
df_sheet2[df_sheet2['Region'].str.contains('Canada', na=False)]

No "Canada" in df_sheet2 but we do have Eastern Canada and Western Canada each with a different person. If they had been the same person then it would be pretty easy - just map everything in either region to "Canada" and merge that way. What we need to do is figure out how to tell if a record in the first sheet is related to eastern or western Canada.

I'm taking a bit of liberty with the data and I'll create a map based on the Canadian Province. If I map it wrong the fix is pretty easy - just change the map.

**Note:** The correct time to fix this is in df _before_ the merge. I'm going to fix it here by nuking the Person column, fixing the data, then re-merging. This is not how I would do it in an actual production situation but it makes the notebook flow better.

In [0]:
canada_region = {
    "Ontario": "Eastern Canada",
    "Quebec": "Eastern Canada",
    "Nova Scotia": "Eastern Canada",
    "Newfoundland": "Eastern Canada",
    "New Brunswick": "Eastern Canada",
    "Prince Edward Island": "Eastern Canada",
    "British Columbia": "Western Canada",
    "Alberta": "Western Canada",
    "Saskatchewan": "Western Canada",
    "Manitoba": "Western Canada",
}

is_ca = df["Country"] == "Canada"
df.loc[is_ca, "Region"] = df.loc[is_ca, "State"].map(canada_region)

Make sure that we fixed all of the Regions - everything should be either Eastern or Western Canada.

In [0]:
df[df['Country'] == 'Canada'].groupby('Region').size().reset_index(name='Count').sort_values('Count', ascending=False)

Need to drop the Person column so that we can merge with the improved Region column

In [0]:
df = df.drop(columns=['Person'])
df.columns

Do the re-merge

In [0]:
df = df.merge(df_sheet2, how='left', on='Region')

We no longer have empty Person fields...

In [0]:
df.isna().sum()

## Review the Data Again

In [0]:
df.head()

In [0]:
df.tail()

In [0]:
df.sample(10)

In [0]:
df.shape

In [0]:
df.info()

In [0]:
df.describe()

In [0]:
# Describe the text (object) fields
df.describe(include=['object'])